# 🏆 Copa Challenger — Dia 1: SQL e Entendimento dos Dados

Objetivo: Explorar a estrutura dos dados, identificar padrões, gerar consultas SQL e produzir insights iniciais.

**Dataset:** Copas do Mundo 2018 e 2022 (128 partidas, 44 colunas)

**Data:** 19/07/2026

In [ ]:
import pandas as pd
import duckdb
import os
from pathlib import Path

# Configurar paths
RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Bibliotecas carregadas')
print(f'📂 Raw dir: {RAW_DIR.resolve()}')

## 1. Carregamento dos Dados

In [ ]:
# Carregar todos os datasets
matches_full = pd.read_csv(RAW_DIR / 'matches_1930_2022.csv')
world_cup = pd.read_csv(RAW_DIR / 'world_cup.csv')
schedule_2026 = pd.read_csv(RAW_DIR / 'schedule_2026.csv')
ranking_2022 = pd.read_csv(RAW_DIR / 'fifa_ranking_2022-10-06.csv')
ranking_2026 = pd.read_csv(RAW_DIR / 'fifa_ranking_2026-06-08.csv')

print(f'📊 matches_1930_2022: {matches_full.shape[0]} partidas, {matches_full.shape[1]} colunas')
print(f'📊 world_cup: {world_cup.shape[0]} edições, {world_cup.shape[1]} colunas')
print(f'📊 schedule_2026: {schedule_2026.shape[0]} jogos, {schedule_2026.shape[1]} colunas')
print(f'📊 ranking_2022: {ranking_2022.shape[0]} times, {ranking_2022.shape[1]} colunas')
print(f'📊 ranking_2026: {ranking_2026.shape[0]} times, {ranking_2026.shape[1]} colunas')

In [ ]:
# Filtrar apenas 2018 e 2022 (regra da competição)
matches = matches_full[matches_full['Year'].isin([2018, 2022])].copy()
matches = matches.reset_index(drop=True)

# Salvar versão filtrada
matches.to_csv(PROCESSED_DIR / 'matches_2018_2022.csv', index=False)

print(f'✅ Dataset filtrado: {len(matches)} partidas (2018+2022)')
print(f'   2018: {len(matches[matches["Year"]==2018])} partidas')
print(f'   2022: {len(matches[matches["Year"]==2022])} partidas')

## 2. Exploração do Schema

In [ ]:
# Schema completo
print('=== SCHEMA ===')
for col in matches.columns:
    dtype = matches[col].dtype
    non_null = matches[col].notna().sum()
    pct = non_null / len(matches) * 100
    nunique = matches[col].nunique()
    print(f'{col:40s} | {str(dtype):8s} | {non_null:3d}/{len(matches)} ({pct:5.1f}%) | {nunique:4d} unique')

In [ ]:
# Primeiras linhas
matches.head()

In [ ]:
# Valores ausentes
missing = matches.isnull().sum()
missing_pct = (missing / len(matches) * 100).round(1)
missing_df = pd.DataFrame({'missing': missing, 'pct': missing_pct})
missing_df = missing_df[missing_df['missing'] > 0].sort_values('pct', ascending=False)
print('=== COLUNAS COM VALORES AUSENTES ===')
print(missing_df.to_string())

## 3. Consultas SQL com DuckDB

In [ ]:
# Criar conexão DuckDB e carregar dados
con = duckdb.connect(':memory:')
con.execute('CREATE TABLE matches AS SELECT * FROM matches')
con.execute('CREATE TABLE ranking_2022 AS SELECT * FROM ranking_2022')
con.execute('CREATE TABLE ranking_2026 AS SELECT * FROM ranking_2026')

print('✅ DuckDB carregado com', con.execute('SELECT count(*) FROM matches').fetchone()[0], 'partidas')

In [ ]:
# SQL 1: Distribuição de resultados por ano
query1 = """
SELECT 
    Year,
    COUNT(*) as total,
    SUM(CASE WHEN home_score > away_score THEN 1 ELSE 0 END) as vitoria_mandante,
    SUM(CASE WHEN home_score = away_score THEN 1 ELSE 0 END) as empate,
    SUM(CASE WHEN home_score < away_score THEN 1 ELSE 0 END) as vitoria_visitante,
    ROUND(AVG(home_score + away_score), 2) as media_gols_por_jogo
FROM matches
GROUP BY Year
ORDER BY Year
"""
result1 = con.execute(query1).fetchdf()
print('=== DISTRIBUIÇÃO DE RESULTADOS POR ANO ===')
print(result1.to_string(index=False))

In [ ]:
# SQL 2: Times com mais jogos e vitórias
query2 = """
WITH team_results AS (
    SELECT home_team as team, Year, home_score as goals_for, away_score as goals_against,
           CASE WHEN home_score > away_score THEN 'W' WHEN home_score = away_score THEN 'D' ELSE 'L' END as result
    FROM matches
    UNION ALL
    SELECT away_team as team, Year, away_score as goals_for, home_score as goals_against,
           CASE WHEN away_score > home_score THEN 'W' WHEN away_score = home_score THEN 'D' ELSE 'L' END as result
    FROM matches
)
SELECT team,
       COUNT(*) as jogos,
       SUM(CASE WHEN result='W' THEN 1 ELSE 0 END) as vitorias,
       SUM(CASE WHEN result='D' THEN 1 ELSE 0 END) as empates,
       SUM(CASE WHEN result='L' THEN 1 ELSE 0 END) as derrotas,
       SUM(goals_for) as gols_feitos,
       SUM(goals_against) as gols_sofridos,
       ROUND(SUM(goals_for)::FLOAT / COUNT(*), 2) as media_gols_jogo
FROM team_results
GROUP BY team
ORDER BY vitorias DESC, gols_feitos DESC
LIMIT 20
"""
result2 = con.execute(query2).fetchdf()
print('=== TOP 20 TIMES (2018+2022) ===')
print(result2.to_string(index=False))

In [ ]:
# SQL 3: Distribuição por fase do torneio
query3 = """
SELECT 
    Round,
    COUNT(*) as jogos,
    ROUND(AVG(home_score + away_score), 2) as media_gols,
    ROUND(AVG(home_score), 2) as media_gols_home,
    ROUND(AVG(away_score), 2) as media_gols_away,
    SUM(CASE WHEN home_score > away_score THEN 1 ELSE 0 END) as home_wins,
    SUM(CASE WHEN home_score = away_score THEN 1 ELSE 0 END) as draws,
    SUM(CASE WHEN home_score < away_score THEN 1 ELSE 0 END) as away_wins
FROM matches
GROUP BY Round
ORDER BY 
    CASE Round
        WHEN 'Group Stage' THEN 1
        WHEN 'Round of 16' THEN 2
        WHEN 'Quarter-finals' THEN 3
        WHEN 'Semi-finals' THEN 4
        WHEN 'Third Place' THEN 5
        WHEN 'Final' THEN 6
        ELSE 7
    END
"""
result3 = con.execute(query3).fetchdf()
print('=== DISTRIBUIÇÃO POR FASE DO TORNEIO ===')
print(result3.to_string(index=False))

In [ ]:
# SQL 4: Gols por minuto (padrões temporais)
query4 = """
SELECT 
    CASE 
        WHEN home_goal LIKE '%0-15%' OR home_goal LIKE '%1\'%'' THEN '0-15'
        WHEN home_goal LIKE '%16-30%' OR home_goal LIKE '%2\'%'' THEN '16-30'
        WHEN home_goal LIKE '%31-45%' OR home_goal LIKE '%3\'%'' OR home_goal LIKE '%4\'%'' THEN '31-45'
        WHEN home_goal LIKE '%46-60%' OR home_goal LIKE '%5\'%'' OR home_goal LIKE '%6\'%'' THEN '46-60'
        WHEN home_goal LIKE '%61-75%' OR home_goal LIKE '%7\'%'' THEN '61-75'
        WHEN home_goal LIKE '%76-90%' OR home_goal LIKE '%8\'%'' OR home_goal LIKE '%9\'%'' THEN '76-90'
        WHEN home_goal LIKE '%90%' OR home_goal LIKE '%1\'0\'%'' OR home_goal LIKE '%1\'1\'%'' OR home_goal LIKE '%1\'2\'%'' THEN '90+'
        ELSE 'N/A'
    END as periodo,
    COUNT(*) as gols_home
FROM matches
WHERE home_goal IS NOT NULL AND home_goal != ''
GROUP BY periodo
ORDER BY gols_home DESC
LIMIT 10
"""
# Consulta mais simples: média de gols por jogo
query4_simple = """
SELECT 
    Year,
    ROUND(AVG(home_score), 2) as media_gols_home,
    ROUND(AVG(away_score), 2) as media_gols_away,
    ROUND(AVG(home_score + away_score), 2) as media_total,
    MAX(home_score + away_score) as max_gols_jogo,
    MIN(home_score + away_score) as min_gols_jogo
FROM matches
GROUP BY Year
ORDER BY Year
"""
result4 = con.execute(query4_simple).fetchdf()
print('=== ESTATÍSTICAS DE GOLS POR ANO ===')
print(result4.to_string(index=False))

In [ ]:
# SQL 5: Confrontos diretos entre seleções
query5 = """
SELECT 
    home_team,
    away_team,
    COUNT(*) as confrontos,
    SUM(CASE WHEN home_score > away_score THEN 1 ELSE 0 END) as home_wins,
    SUM(CASE WHEN home_score = away_score THEN 1 ELSE 0 END) as draws,
    SUM(CASE WHEN home_score < away_score THEN 1 ELSE 0 END) as away_wins
FROM matches
GROUP BY home_team, away_team
HAVING COUNT(*) >= 2
ORDER BY confrontos DESC
LIMIT 15
"""
result5 = con.execute(query5).fetchdf()
print('=== CONFRONTOS DIRETOS FREQUENTES ===')
print(result5.to_string(index=False))

## 4. Cruzamento com Rankings FIFA

In [ ]:
# Verificar estrutura dos rankings
print('=== RANKING 2022 (primeiras 10 linhas) ===')
print(ranking_2022.head(10).to_string())
print()
print('=== COLUNAS RANKING 2022 ===')
print(ranking_2022.columns.tolist())

In [ ]:
# SQL 6: Cruzar ranking com resultados
query6 = """
SELECT 
    m.Year,
    m.home_team,
    m.away_team,
    r1.rank as rank_home,
    r1.points as points_home,
    r2.rank as rank_away,
    r2.points as points_away,
    r1.rank - r2.rank as rank_diff,
    r1.points - r2.points as points_diff,
    m.home_score,
    m.away_score,
    CASE WHEN m.home_score > m.away_score THEN 'H' 
         WHEN m.home_score = m.away_score THEN 'D' 
         ELSE 'A' END as result
FROM matches m
LEFT JOIN ranking_2022 r1 ON m.home_team = r1.team AND m.Year = 2022
LEFT JOIN ranking_2022 r2 ON m.away_team = r2.team AND m.Year = 2022
WHERE m.Year = 2022
UNION ALL
SELECT 
    m.Year,
    m.home_team,
    m.away_team,
    r1.rank as rank_home,
    r1.points as points_home,
    r2.rank as rank_away,
    r2.points as points_away,
    r1.rank - r2.rank as rank_diff,
    r1.points - r2.points as points_diff,
    m.home_score,
    m.away_score,
    CASE WHEN m.home_score > m.away_score THEN 'H' 
         WHEN m.home_score = m.away_score THEN 'D' 
         ELSE 'A' END as result
FROM matches m
LEFT JOIN ranking_2026 r1 ON m.home_team = r1.team AND m.Year = 2018
LEFT JOIN ranking_2026 r2 ON m.away_team = r2.team AND m.Year = 2018
WHERE m.Year = 2018
"""
result6 = con.execute(query6).fetchdf()
print(f'✅ Cruzamento ranking×resultado: {len(result6)} registros')
print(f'   Com dados de ranking: {result6["rank_home"].notna().sum()} ({result6["rank_home"].notna().sum()/len(result6)*100:.1f}%)')
print()
# Análise: diferença de ranking vs resultado
if result6['rank_diff'].notna().sum() > 0:
    valid = result6.dropna(subset=['rank_diff'])
    print('=== DIFERENÇA DE RANKING vs RESULTADO ===')
    for r in ['H', 'D', 'A']:
        subset = valid[valid['result'] == r]
        if len(subset) > 0:
            print(f'Resultado {r}: rank_diff médio = {subset["rank_diff"].mean():.1f}, n = {len(subset)}')

## 5. Análise de Features Potenciais

In [ ]:
# Preparar features para análise
df = matches.copy()

# Feature: resultado
df['result'] = df.apply(lambda r: 'H' if r['home_score'] > r['away_score'] else ('D' if r['home_score'] == r['away_score'] else 'A'), axis=1)

# Feature: total de gols
df['total_goals'] = df['home_score'] + df['away_score']

# Feature: diferença de gols
df['goal_diff'] = df['home_score'] - df['away_score']

# Feature: foi para prorrogação?
df['had_extra_time'] = df['Notes'].str.contains('extra time', case=False, na=False).astype(int)

# Feature: foi para pênaltis?
df['had_penalties'] = df['Notes'].str.contains('penalty', case=False, na=False).astype(int)

print('=== FEATURES CRIADAS ===')
print(f'result: {df["result"].value_counts().to_dict()}')
print(f'total_goals: média={df["total_goals"].mean():.2f}, std={df["total_goals"].std():.2f}')
print(f'had_extra_time: {df["had_extra_time"].sum()} jogos ({df["had_extra_time"].mean()*100:.1f}%)')
print(f'had_penalties: {df["had_penalties"].sum()} jogos ({df["had_penalties"].mean()*100:.1f}%)')

In [ ]:
# Análise: xG (expected goals) disponível?
print('=== COLUNAS xG ===')
xg_cols = [c for c in df.columns if 'xg' in c.lower()]
print(f'Colunas xG encontradas: {xg_cols}')

if 'home_xg' in df.columns and 'away_xg' in df.columns:
    print(f'home_xg: {df["home_xg"].notna().sum()}/{len(df)} ({df["home_xg"].notna().mean()*100:.1f}%)')
    print(f'away_xg: {df["away_xg"].notna().sum()}/{len(df)} ({df["away_xg"].notna().mean()*100:.1f}%)')
    
    # xG vs gols reais
    valid_xg = df.dropna(subset=['home_xg', 'away_xg'])
    if len(valid_xg) > 0:
        print(f'\nMédia home_xg: {valid_xg["home_xg"].mean():.2f} vs home_score real: {valid_xg["home_score"].mean():.2f}')
        print(f'Média away_xg: {valid_xg["away_xg"].mean():.2f} vs away_score real: {valid_xg["away_score"].mean():.2f}')
        print(f'Correlação home_xg × home_score: {valid_xg["home_xg"].corr(valid_xg["home_score"]):.3f}')
        print(f'Correlação away_xg × away_score: {valid_xg["away_xg"].corr(valid_xg["away_score"]):.3f}')

In [ ]:
# Análise: Attendance (público)
print('=== PÚBLICO ===')
if 'Attendance' in df.columns:
    att = df['Attendance'].dropna()
    print(f'Jogos com dados de público: {len(att)}/{len(df)}')
    print(f'Média: {att.mean():.0f}')
    print(f'Mediana: {att.median():.0f}')
    print(f'Mín: {att.min():.0f}, Máx: {att.max():.0f}')
    
    # Público vs resultado
    df_att = df.dropna(subset=['Attendance'])
    print(f'\nMédia público - Vitoria mandante: {df_att[df_att["result"]=="H"]["Attendance"].mean():.0f}')
    print(f'Média público - Empate: {df_att[df_att["result"]=="D"]["Attendance"].mean():.0f}')
    print(f'Média público - Vitoria visitante: {df_att[df_att["result"]=="A"]["Attendance"].mean():.0f}')

In [ ]:
# Análise: Cartões (disciplina)
print('=== DISCIPLINA ===')
card_cols = [c for c in df.columns if 'card' in c.lower()]
print(f'Colunas de cartões: {card_cols}')

# Contar cartões amarelos e vermelhos
for col in ['home_yellow_card_long', 'away_yellow_card_long', 'home_red_card', 'away_red_card']:
    if col in df.columns:
        count = df[col].notna().sum()
        print(f'{col}: {count}/{len(df)} jogos ({count/len(df)*100:.1f}%)')

## 6. Insights Iniciais

In [ ]:
# Resumo dos insights
print('=' * 60)
print('📋 RESUMO DOS INSIGHTS — DIA 1')
print('=' * 60)
print()
print(f'1. DATASET: {len(matches)} partidas (2018+2022), {matches.shape[1]} colunas')
print(f'   - {len(matches[matches["Year"]==2018])} em 2018, {len(matches[matches["Year"]==2022])} em 2022')
print()
print(f'2. RESULTADOS:')
hw = len(matches[matches['result']=='H'])
dr = len(matches[matches['result']=='D'])
aw = len(matches[matches['result']=='A'])
print(f'   - Vitória mandante: {hw} ({hw/len(matches)*100:.1f}%)')
print(f'   - Empate: {dr} ({dr/len(matches)*100:.1f}%)')
print(f'   - Vitória visitante: {aw} ({aw/len(matches)*100:.1f}%)')
print()
print(f'3. GOLS:')
print(f'   - Média por jogo: {matches["total_goals"].mean():.2f}')
print(f'   - Máximo em um jogo: {matches["total_goals"].max()}')
print(f'   - Jogos com 0 gols: {len(matches[matches["total_goals"]==0])} ({len(matches[matches["total_goals"]==0])/len(matches)*100:.1f}%)')
print()
print(f'4. FEATURES MAIS PROMISSORAS (baseado na exploração):')
print(f'   - home_xg / away_xg (se disponíveis)')
print(f'   - Diferença de ranking FIFA')
print(f'   - Fase do torneio (Round)')
print(f'   - Mando de campo')
print(f'   - Histórico de confrontos diretos')
print()
print(f'5. GAPS IDENTIFICADOS:')
missing_count = matches.isnull().sum()
high_missing = missing_count[missing_count > len(matches)*0.3]
if len(high_missing) > 0:
    print(f'   - Colunas com >30% ausentes: {high_missing.index.tolist()}')
print(f'   - Dados de ranking FIFA podem não cobrir todos os confrontos')

In [ ]:
# Salvar dataset processado
df.to_csv(PROCESSED_DIR / 'matches_2018_2022_features.csv', index=False)
print(f'✅ Dataset com features salvo em: {PROCESSED_DIR / "matches_2018_2022_features.csv"}')

# Fechar DuckDB
con.close()
print('✅ DuckDB fechado')